# Document Processing and Chunking## Why Chunking MattersLLMs have token limits. Documents must be split into chunks!### Challenges- Chunk too small → lose context- Chunk too large → exceed token limits- Poor splits → break semantic meaning### Strategies1. Fixed-size chunks2. Sentence-based3. Paragraph-based4. Semantic chunking5. Recursive chunking

In [ ]:
import numpy as npimport pandas as pdfrom typing import List, Dict, Tupleimport jsonimport osfrom pathlib import Pathfrom langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

## Strategy 1: Fixed-Size ChunksSimplest approach - split by character count

In [ ]:
text = """Artificial intelligence has transformed many industries. Machine learning algorithms can now process vast amounts of data. Deep learning networks achieve human-level performance in many tasks.Natural language processing enables computers to understand human language.Computer vision allows machines to interpret visual information."""def fixed_size_chunk(text: str, chunk_size: int = 100, overlap: int = 20):    """Split text into fixed-size chunks with overlap."""    chunks = []    start = 0    while start < len(text):        end = start + chunk_size        chunks.append(text[start:end])        start = end - overlap    return chunkschunks = fixed_size_chunk(text)for i, chunk in enumerate(chunks, 1):    print(f"Chunk {i}: {chunk}...")    print()

## Strategy 2: Sentence-BasedSplit at sentence boundaries

In [ ]:
import nltk# nltk.download('punkt')  # Run oncedef sentence_chunk(text: str, sentences_per_chunk: int = 2):    """Split by sentences."""    sentences = nltk.sent_tokenize(text)    chunks = []    for i in range(0, len(sentences), sentences_per_chunk):        chunk = ' '.join(sentences[i:i+sentences_per_chunk])        chunks.append(chunk)    return chunks, sentenceschunks, sentences = sentence_chunk(text)print(f"Total sentences: {len(sentences)}")print(f"Total chunks: {len(chunks)}\n")for i, chunk in enumerate(chunks, 1):    print(f"Chunk {i}: {chunk}")    print()

## Strategy 3: Recursive ChunkingTry multiple separators hierarchically

In [ ]:
splitter = RecursiveCharacterTextSplitter(    chunk_size=150,    chunk_overlap=30,    separators=["\n\n", "\n", ". ", " ", ""])long_text = """Retrieval-Augmented Generation (RAG) is a powerful technique.It combines two key components:1. Information Retrieval2. Text GenerationThe retrieval step searches a knowledge base.The generation step uses an LLM to create responses.This approach has several advantages.It reduces hallucinations and provides source attribution."""chunks = splitter.split_text(long_text)print(f"Created {len(chunks)} chunks:\n")for i, chunk in enumerate(chunks, 1):    print(f"Chunk {i} ({len(chunk)} chars):")    print(chunk)    print("-" * 50)

## Strategy 4: Semantic ChunkingSplit based on meaning, not just length

In [ ]:
from sentence_transformers import SentenceTransformermodel = SentenceTransformer('all-MiniLM-L6-v2')def semantic_chunk(text: str, threshold: float = 0.5):    """Chunk based on semantic similarity."""    sentences = nltk.sent_tokenize(text)        if len(sentences) <= 1:        return [text]        # Get embeddings    embeddings = model.encode(sentences)        # Compute similarities    chunks = []    current_chunk = [sentences[0]]        for i in range(1, len(sentences)):        # Similarity with previous        sim = np.dot(embeddings[i], embeddings[i-1])        sim /= (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i-1]))                if sim > threshold:            current_chunk.append(sentences[i])        else:            chunks.append(' '.join(current_chunk))            current_chunk = [sentences[i]]        chunks.append(' '.join(current_chunk))    return chunkssemantic_chunks = semantic_chunk(text, threshold=0.6)print(f"Semantic chunks: {len(semantic_chunks)}\n")for i, chunk in enumerate(semantic_chunks, 1):    print(f"Semantic Chunk {i}:")    print(chunk)    print()

## Best Practices### Chunk Size Guidelines| Use Case | Chunk Size | Overlap ||----------|-----------|----------|| Short Q&A | 200-500 | 50-100 || Documents | 500-1000 | 100-200 || Code | 1000-2000 | 200-300 |### Tips✅ Keep context intact (don't split mid-sentence)✅ Add overlap to preserve context across chunks✅ Include metadata (source, page number, etc.)✅ Test different strategies for your domain✅ Monitor retrieval quality**Next**: Using LangChain for production RAG!